# 05. Строительные блоки: patches, time, AdaLN и gates

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** воспроизвести patch layout и проверить native modulation/residual на маленьких тензорах

**Перед началом:** K6 01–03; linear layers и broadcasting. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/03_nn_building_blocks.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §3.2](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/dit.py)
- [`core/tensors.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/tensors.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','main'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '05_nn_patches_and_modulation'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Из локальной ячейки latent в visual token

Visual input содержит state, conditioning channels и mask: 16+16+1=33.
Patch 1×2×2 собирает 4 ячейки в вектор длины 132, затем Linear переводит его в width 1792.
Audio tokens не проходят spatial patching: 40→896. В tiny опыте ширины меньше, операции те же.

Unpatchify в output голове имеет собственный порядок C/patch axes. Нельзя предположить,
что inverse input permutation восстанавливает output без проверки порядка выходных features.


In [ ]:
D=native();x=torch.arange(1*2*4*4*3,dtype=torch.float32).reshape(1,2,4,4,3)
layer=D.VisualEmbeddings(3,12,(1,2,2))
with torch.no_grad():layer.in_layer.weight.copy_(torch.eye(12));layer.in_layer.bias.zero_()
y=layer(x)
manual=x.reshape(1,2,1,2,2,2,2,3).permute(0,1,3,5,2,4,6,7).flatten(4,7)
assert torch.equal(y,manual)
print('Native patches:',tuple(y.shape),'first vector:',y[0,0,0,0])
metrics['visual_patch_max_error']=(y-manual).abs().max().item()


## 2. AdaLN — condition управляет обработкой state

Time sinusoid проходит MLP. CLIP pooled projection прибавляется к time representation.
Modulation выдаёт shift, scale и residual gate:

$$
h' = h + g(c) F\left((1+s(c))\mathrm{LN}(h)+b(c)\right).
$$

При нулевом gate residual branch не меняет h. Это свойство forward; градиенты gate всё ещё могут
быть ненулевыми. Нельзя сказать «нулевой слой никогда не учится». Если занулить и предшествующее
представление, градиент может исчезнуть по другой причине.


In [ ]:
tensors=native('core.tensors');h=torch.randn(2,5,8);condition=torch.randn(2,6)
mod=D.Modulation(6,8,3);shift,scale,gate=mod(condition).chunk(3,-1)
branch=nn.Linear(8,8);norm=nn.LayerNorm(8,elementwise_affine=False)
out=tensors.apply_gate_sum(h,branch(tensors.apply_scale_shift_norm(norm,h,scale,shift)),gate)
identity=(out-h).abs().max().item();assert identity==0
out.square().mean().backward()
metrics['zero_gate_identity_error']=identity
metrics['modulation_gradient_norm']=mod.out_layer.weight.grad.norm().item()
assert metrics['modulation_gradient_norm']>0
print(metrics)


## 3. OutLayerAudio и маленькие на вид особенности

Audio output выполняет modulated LayerNorm, затем ещё одну LayerNorm, затем Linear.
Это буквальное поведение pinned source. Удалять второй norm «для красоты» при переносе нельзя:
проверка parity должна сравнивать именно вычисляемую функцию.


In [ ]:
audio=D.OutLayerAudio(8,6,4);a=torch.randn(1,5,8);t=torch.randn(1,1,6)
with torch.no_grad():
    actual=audio(a,t)
    shift,scale=audio.modulation(t).chunk(2,-1)
    m=tensors.apply_scale_shift_norm(audio.norm,a,scale,shift)
    expected=audio.out_layer(audio.norm(m))
assert torch.allclose(actual,expected)
metrics['audio_output_manual_max_error']=(actual-expected).abs().max().item()
print('Audio output shape:',actual.shape)


## Самостоятельное исследование

1. Посчитайте input projection dimensions для 16-channel и 33-channel входа.
2. Сравните gradient modulation и gradient residual branch на первом zero-init шаге.
3. Найдите в DiT FFN две bias-free Linears и GELU; посчитайте их параметры вручную.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы прослеживаете формы от latent cells до tokens и обратно, не смешивая число heads,
model width, latent channels и число grid points. Нулевой gate и frozen parameters — разные понятия.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 04. Аудио: waveform → mel → VAE → vocoder](04_audio_vae_and_vocoder.ipynb) · [06. Attention и RoPE: текст, видео и аудио →](06_attention_and_rope.ipynb)
